# N164 · 线程、互斥与顺序同步

**目标：** 用同步关系保证顺序而不是依靠启动次序。

**先修：** N009, N053, N013。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** threading；共享状态；race；Lock/Event；happens-before；GIL非同步契约。

## 从问题到算法

启动顺序不是执行顺序。即使先启动 third 线程，也必须等 second 的回调完成；这是一条执行先后关系，而不是“睡一会儿大概够了”。

first 在回调结束后设置事件 A，second 等 A 后运行并设置 B，third 等 B。事件保留已触发状态，因此先触发再等待也不会丢失通知。

测试实际启动线程，并把整组可能阻塞的线程放进独立进程。外层超时会终止并回收子进程，同时使 Notebook 测试失败；没有把超时算成成功，也不使用 daemon 线程掩盖泄漏。

## 最小实现

**本章接口：** `Foo.first/second/third`、`run_ordering_case(start_order)`

In [1]:
THREAD_SOURCE = r'''import threading
from collections import deque

def run_threads(functions):
    """Caller must run potentially blocking cases inside the isolated process."""
    errors = []
    error_lock = threading.Lock()
    def guarded(fn):
        try:
            fn()
        except BaseException as error:
            with error_lock:
                errors.append(error)
    threads = [threading.Thread(target=guarded, args=(fn,)) for fn in functions]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    if errors:
        raise errors[0]
    assert not any(thread.is_alive() for thread in threads)

class Foo:
    def __init__(self):
        self.first_done = threading.Event()
        self.second_done = threading.Event()
    def first(self, printFirst):
        printFirst()
        self.first_done.set()
    def second(self, printSecond):
        self.first_done.wait()
        printSecond()
        self.second_done.set()
    def third(self, printThird):
        self.second_done.wait()
        printThird()

def run_ordering_case(start_order):
    if sorted(start_order) != [0,1,2]:
        raise ValueError('Each method must be launched exactly once')
    foo = Foo()
    out = []
    methods = [lambda: foo.first(lambda: out.append('first')),
               lambda: foo.second(lambda: out.append('second')),
               lambda: foo.third(lambda: out.append('third'))]
    run_threads([methods[i] for i in start_order])
    return ''.join(out)
'''

import json
import subprocess
import sys

# This defines the visible APIs only; it does not launch a thread.
exec(THREAD_SOURCE, globals())

def run_isolated(extra):
    """A timeout is a failure; subprocess.run kills and reaps the child."""
    result = subprocess.run([sys.executable, '-c', THREAD_SOURCE + '\n' + extra],
        text=True, capture_output=True, check=True, timeout=20)
    return result.stdout

## 正确性、前提与复杂度

A 只能在 first 回调之后触发，second 只能在 A 之后运行；同理 third 在 second 之后。因此由传递性保证完整顺序，各方法启动一次保证回调各一次。活性另要求所有方法都被调用、回调有限完成、线程最终获得调度。

**代价：** 同步状态 O(1)。等待时间由任务和调度决定，不应声称固定毫秒延迟；调度测试有限次通过不等于证明所有环境都及时响应。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

from IPython.display import Code
display(Code(THREAD_SOURCE,language='python'))
rows = json.loads(run_isolated("import json,itertools; print(json.dumps([[list(p),run_ordering_case(p)] for p in itertools.permutations(range(3))]))"))
show_table(['thread start order','actual callback order'],rows)

import threading
from collections import deque

def run_threads(functions):
    """Caller must run potentially blocking cases inside the isolated process."""
    errors = []
    error_lock = threading.Lock()
    def guarded(fn):
        try:
            fn()
        except BaseException as error:
            with error_lock:
                errors.append(error)
    threads = [threading.Thread(target=guarded, args=(fn,)) for fn in functions]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    if errors:
        raise errors[0]
    assert not any(thread.is_alive() for thread in threads)

class Foo:
    def __init__(self):
        self.first_done = threading.Event()
        self.second_done = threading.Event()
    def first(self, printFirst):
        printFirst()
        self.first_done.set()
    def second(self, printSecond):
        self.first_done.wait()
        printSecond()
        self.second_done.set()
    def third(self, printThird):
        self.second_done.wait()
        printThird()

def run_ordering_case(start_order):
    if sorted(start_order) != [0,1,2]:
        raise ValueError('Each method must be launched exactly once')
    foo = Foo()
    out = []
    methods = [lambda: foo.first(lambda: out.append('first')),
               lambda: foo.second(lambda: out.append('second')),
               lambda: foo.third(lambda: out.append('third'))]
    run_threads([methods[i] for i in start_order])
    return ''.join(out)

thread start order,actual callback order
"[0, 1, 2]",firstsecondthird
"[0, 2, 1]",firstsecondthird
"[1, 0, 2]",firstsecondthird
"[1, 2, 0]",firstsecondthird
"[2, 0, 1]",firstsecondthird
"[2, 1, 0]",firstsecondthird


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
print(run_isolated("""import itertools
for _ in range(10):
    for order in itertools.permutations(range(3)):
        assert run_ordering_case(order) == 'firstsecondthird'
assert threading.active_count() == 1
print('All six start orders passed; every worker joined.')
"""))
print('N164: 所有本章断言通过')

All six start orders passed; every worker joined.

N164: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 枚举三线程启动排列。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 构造无同步版本的可能交错，而不要求随机必现错误。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1114. Print in Order（canonical）：[官方入口](https://leetcode.com/problems/print-in-order/)

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。